# Predicting Customer Churn in Telecommunications Using Supervised Machine Learning

## 1. Introduction and business rationale

Customer churn occurs when a customer ends their relationship with a service provider. In telecommunications, unexpected departures can reduce recurring revenue and increase the need to acquire replacement customers. Identifying customers who may leave gives a provider an opportunity to investigate dissatisfaction and offer relevant support or retention incentives.

This project investigates whether customer account, demographic and service information can predict churn. The task is **binary supervised classification**: the target is whether a customer churned (`Yes`) or remained (`No`). Two approaches are compared:

- **Logistic Regression:** a regularised linear classification baseline.
- **Random Forest:** a tree-ensemble method that can represent non-linear relationships and interactions.

The objective is not simply to maximise accuracy. Because churners are the minority class, the analysis also considers **precision, recall, F1-score and ROC-AUC**. Recall measures the proportion of actual churners identified; precision measures how many customers flagged as likely to churn actually churned. The preferred operating point depends on the cost of missed churners and unnecessary retention contacts.

### Dataset

The Telco Customer Churn dataset contains 7,043 customer records and demographic, account and subscribed-service variables. `Churn` is the binary outcome. `customerID` is an identifier and is excluded from modelling because it does not represent a meaningful customer characteristic.

**Data source:** IBM sample Telco Customer Churn data, distributed through Kaggle (see References). The dataset represents a particular provider and should not be assumed to describe the entire telecommunications market.

## 2. Data loading and initial inspection

The CSV file should be included in the same folder as this notebook in the submission ZIP. The code checks that the expected file and target column are present. Blank strings in `TotalCharges` are converted to missing values; imputation is postponed until after the train-test split and is performed inside the modelling pipelines.

In [ ]:
# 2.1 Imports and reproducibility
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("Telco_Customer_Churn.csv")
sns.set_theme(style="whitegrid", context="notebook")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Put the CSV in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
required_columns = {"Churn", "customerID", "TotalCharges", "Contract"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {sorted(missing_columns)}")

# Convert blank TotalCharges entries to NaN; do not calculate an imputation value here.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True),
    errors="coerce"
)

print(f"Dataset dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print("\nData types and non-null counts:")
df.info()
display(df.head())

### Initial data-quality assessment

The inspection below checks target labels, missing values and duplicate records. Missing `TotalCharges` values are expected for some customers with very short tenure. They are not filled here: the imputer learns its median from each training fold, preventing information from validation or test data entering model fitting.

In [ ]:
# 2.2 Data quality checks
print("Target values:")
print(df["Churn"].value_counts(dropna=False))

print("\nMissing values by column (largest first):")
missing_summary = df.isna().sum().loc[lambda x: x.gt(0)].sort_values(ascending=False)
display(missing_summary.to_frame("missing_count"))

print(f"\nMissing TotalCharges values: {df['TotalCharges'].isna().sum()}")
print(f"Duplicate customer IDs: {df['customerID'].duplicated().sum()}")

if not set(df["Churn"].dropna().unique()).issubset({"Yes", "No"}):
    raise ValueError("Unexpected values found in Churn; inspect target labels before modelling.")

## 3. Exploratory data analysis

Figure 1 summarises the target distribution and compares observed churn rates across contract types. The target distribution matters because a model can achieve apparently high accuracy by favouring the majority class. Contract-level rates provide a descriptive indication of whether churn differs across customer groups; they are associations, not evidence that contract type causes churn.

In [ ]:
# Figure 1: target distribution and descriptive churn rate by contract
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
sns.barplot(
    x=churn_counts.index, y=churn_counts.values,
    hue=churn_counts.index, legend=False, ax=axes[0], palette="Set2"
)
axes[0].set(title="Customer churn distribution", xlabel="Churn", ylabel="Number of customers")
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.0f")

contract_churn = (
    df.groupby("Contract", observed=False)["Churn"]
      .apply(lambda values: (values == "Yes").mean())
      .sort_values(ascending=False)
)
sns.barplot(
    x=contract_churn.index, y=contract_churn.values,
    hue=contract_churn.index, legend=False, ax=axes[1], palette="Blues"
)
axes[1].set(
    title="Observed churn rate by contract",
    xlabel="Contract type", ylabel="Customers who churned"
)
axes[1].tick_params(axis="x", rotation=15)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

plt.tight_layout()
plt.show()

print("Overall churn rate: {:.1%}".format((df["Churn"] == "Yes").mean()))
display(contract_churn.rename("churn_rate").to_frame())

**Interpretation:** The left panel shows the class imbalance that makes accuracy alone insufficient. The right panel shows how the observed churn rate varies by contract type in this sample. These descriptive patterns motivate classification, but they must be assessed alongside other variables and on held-out data. No causal conclusion should be drawn from this chart.

## 4. Train-test split and leakage-safe preprocessing

The data is split before any imputation, encoding or scaling. Stratification preserves approximately the same churn proportion in the training and test sets. The test set is held back until final evaluation.

All learned preprocessing is placed inside scikit-learn pipelines. Numeric variables are median-imputed; categorical variables are imputed with their most frequent training value and one-hot encoded. Logistic Regression additionally uses feature scaling. Cross-validation therefore fits preprocessing separately within each training fold and prevents test-set information leaking into model development.

In [ ]:
# 4.1 Separate predictors and target, then split raw data
model_df = df.drop(columns=["customerID"]).copy()
X = model_df.drop(columns=["Churn"])
y = model_df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Training observations: {len(X_train):,}")
print(f"Test observations: {len(X_test):,}")
print(f"Numeric predictors ({len(numeric_features)}): {numeric_features}")
print(f"Categorical predictors ({len(categorical_features)}): {categorical_features}")
print(f"Training churn rate: {y_train.mean():.1%}")
print(f"Test churn rate: {y_test.mean():.1%}")

def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

## 5. Model development and hyperparameter tuning

Both models are trained on the same training partition and assessed on the same untouched test partition.

Logistic Regression is used as a baseline. Its regularisation strength (`C`) is selected using stratified cross-validation. Random Forest is tuned over a compact grid of tree count, depth and minimum split size. The search optimises mean cross-validated ROC-AUC, which evaluates ranking quality across probability thresholds rather than relying on one arbitrary threshold.

The grid is restricted to keep runtime appropriate for JupyterHub. The test set is not used to select hyperparameters.

In [ ]:
# 5.1 Logistic Regression pipeline and cross-validated tuning
lr_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("scaler", StandardScaler(with_mean=False)),
    ("classifier", LogisticRegression(
        max_iter=2000, solver="liblinear", random_state=RANDOM_STATE
    ))
])

lr_grid = {"classifier__C": [0.1, 1.0, 10.0]}
lr_search = GridSearchCV(
    estimator=lr_pipeline, param_grid=lr_grid, scoring="roc_auc",
    cv=cv, n_jobs=-1, refit=True
)
lr_search.fit(X_train, y_train)

print("Logistic Regression best parameters:", lr_search.best_params_)
print(f"Logistic Regression mean CV ROC-AUC: {lr_search.best_score_:.4f}")

In [ ]:
# 5.2 Random Forest pipeline and cross-validated tuning
rf_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
    ))
])

rf_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10],
    "classifier__min_samples_split": [2, 10]
}
rf_search = GridSearchCV(
    estimator=rf_pipeline, param_grid=rf_grid, scoring="roc_auc",
    cv=cv, n_jobs=-1, refit=True
)
rf_search.fit(X_train, y_train)

print("Random Forest best parameters:", rf_search.best_params_)
print(f"Random Forest mean CV ROC-AUC: {rf_search.best_score_:.4f}")

## 6. Held-out model evaluation

The selected models are evaluated on the test set, which was not used for preprocessing, cross-validation or hyperparameter selection. The default classification threshold is 0.5. This provides a consistent baseline comparison; a later section explains why a business may choose a different threshold.

ROC-AUC measures ranking discrimination across thresholds. Precision, recall and F1-score describe performance at the chosen threshold. Average precision provides an additional summary of the precision-recall trade-off, useful when the positive class is less common.

In [ ]:
# 6.1 Predictions and probability estimates
models = {
    "Logistic Regression": lr_search.best_estimator_,
    "Tuned Random Forest": rf_search.best_estimator_
}

predictions = {}
probabilities = {}
metric_rows = []

for model_name, fitted_model in models.items():
    y_pred = fitted_model.predict(X_test)
    y_prob = fitted_model.predict_proba(X_test)[:, 1]
    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    metric_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (churn)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (churn)": recall_score(y_test, y_pred, zero_division=0),
        "F1 (churn)": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Average precision": average_precision_score(y_test, y_prob)
    })

results = pd.DataFrame(metric_rows).set_index("Model")
display(results.round(3))

for model_name in models:
    print(f"\n{model_name} — classification report")
    print(classification_report(
        y_test, predictions[model_name],
        target_names=["Retained", "Churned"], digits=3, zero_division=0
    ))

### Interpreting the metrics

- **Accuracy** is the overall proportion of correct predictions. It can mislead when churn is less common than retention.
- **Precision for churn** is the proportion of customers predicted to churn who actually churned. Low precision can create unnecessary retention contacts.
- **Recall for churn** is the proportion of actual churners identified. Low recall means potential churners are missed.
- **F1-score** is the harmonic mean of precision and recall.
- **ROC-AUC** measures ranking discrimination over all possible thresholds; 0.5 represents random ranking and 1.0 represents perfect ranking.
- **Average precision** summarises the precision-recall curve and complements ROC-AUC when the positive class is less frequent.

The strongest model should not be selected from accuracy alone. The business objective and the relative costs of false negatives and false positives should guide the final choice.

## 7. Error analysis and model comparison

The confusion matrices show true negatives, false positives, false negatives and true positives at the default threshold. Comparing both models side by side makes the operational trade-off clearer. The ROC curves compare probability ranking across thresholds.

In [ ]:
# Figure 2: Confusion matrices for both models
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, model_name in zip(axes, models):
    cm = confusion_matrix(y_test, predictions[model_name], labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=["Retained", "Churned"]
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(model_name)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")

fig.suptitle("Figure 2: Test-set confusion matrices (threshold = 0.50)")
plt.tight_layout()
plt.show()

# Figure 3: ROC curve comparison
plt.figure(figsize=(7, 5))
for model_name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_value = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random ranking")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Figure 3: ROC curves on the held-out test set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Interpretation:** Read the confusion matrices together with the metric table. A false negative is a customer who churned but was predicted to stay; a false positive is a customer predicted to churn who stayed. In a retention campaign, false negatives may represent missed intervention opportunities, while false positives may consume limited contact or discount budgets. The ROC curve does not identify a single best operating threshold; that requires a business cost model.

## 8. Understanding influential predictors

Permutation importance estimates how much a fitted model's test-set ROC-AUC decreases when one original input feature is randomly shuffled. A larger positive decrease indicates that the model relies more heavily on that feature in this evaluation.

This analysis is performed on the tuned Random Forest pipeline and held-out test data. It measures predictive reliance, not causality. Correlated predictors can share or mask importance, so the ranking is not proof that changing a feature will change churn.

In [ ]:
# Figure 4: Permutation importance at the original-feature level
best_rf_model = rf_search.best_estimator_

perm = permutation_importance(
    best_rf_model, X_test, y_test, scoring="roc_auc",
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)

importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Mean AUC decrease": perm.importances_mean,
    "Standard deviation": perm.importances_std
}).sort_values("Mean AUC decrease", ascending=False)

top_importance = importance.head(10).sort_values("Mean AUC decrease")

plt.figure(figsize=(8, 5))
plt.barh(
    top_importance["Feature"], top_importance["Mean AUC decrease"],
    xerr=top_importance["Standard deviation"], capsize=3
)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Decrease in test ROC-AUC after permutation")
plt.ylabel("Original predictor")
plt.title("Figure 4: Random Forest permutation importance")
plt.tight_layout()
plt.show()

display(importance.head(10).round(4))

## 9. Threshold and commercial implications

A probability threshold of 0.5 is a convention, not necessarily the most profitable decision rule. Lowering the threshold generally identifies more potential churners (higher recall) but can also increase false positives. Raising it generally reduces the number of customers contacted but may miss more churners.

The following table illustrates the trade-off for the tuned Random Forest at several thresholds. It does not identify the most profitable threshold because the dataset does not provide retention-offer costs, customer lifetime value or the probability that an intervention prevents churn.

In [ ]:
# 9.1 Threshold sensitivity for the tuned Random Forest
rf_test_prob = probabilities["Tuned Random Forest"]
threshold_rows = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    threshold_pred = (rf_test_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, threshold_pred, labels=[0, 1]).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, threshold_pred, zero_division=0),
        "Recall": recall_score(y_test, threshold_pred, zero_division=0),
        "F1": f1_score(y_test, threshold_pred, zero_division=0),
        "False positives": fp,
        "False negatives": fn,
        "Customers flagged": int(threshold_pred.sum())
    })

threshold_results = pd.DataFrame(threshold_rows).set_index("Threshold")
display(threshold_results.round(3))

**Commercial interpretation:** A telecom operator could use predicted churn probabilities to prioritise human review or tailored service interventions. The threshold should be chosen using an explicit estimate of the cost of contacting a customer, the expected value of retaining that customer and the effectiveness of the intervention. A model prediction alone does not demonstrate that an offer will prevent churn. Any deployment should also consider fairness, customer privacy, monitoring for changes in behaviour and periodic model revalidation.

## 10. Conclusion and critical evaluation

This project developed and compared Logistic Regression and a tuned Random Forest for predicting customer churn. The models were evaluated using a stratified hold-out test set, with preprocessing fitted inside pipelines and hyperparameters selected using stratified cross-validation. This avoids the test-set contamination that occurs when imputation and encoding are performed before splitting.

Use the results table to identify the stronger model for the intended objective. ROC-AUC compares ranking performance across thresholds, while precision, recall and F1-score describe performance at the default 0.5 threshold. If Logistic Regression has the higher ROC-AUC, it provides the stronger ranking result in this experiment; if Random Forest has the higher recall, it identifies a larger share of churners at the default threshold. The final choice should reflect the business cost of missed churners versus unnecessary retention activity, rather than model complexity or accuracy alone.

Permutation importance indicates which original predictors the Random Forest relies on most. These are predictive associations, not causal drivers. Correlated variables may affect the ranking, and the results should not be used to claim that changing a particular customer characteristic will prevent churn.

### Limitations and future work

1. **Single dataset and provider:** The data represents one provider and may not generalise to other markets, customer groups or time periods.
2. **Single hold-out evaluation:** A separate test set gives an independent evaluation for this experiment, but temporal or external validation would provide stronger evidence of stability.
3. **Class imbalance:** Stratification and class-weighted Random Forest are used, but threshold selection or cost-sensitive learning could be assessed. Resampling should occur only inside training folds.
4. **Limited commercial variables:** The dataset does not provide intervention costs, customer lifetime value, contact history or evidence that a retention offer changes behaviour. Profitability cannot be established from predictions alone.
5. **Predictive rather than causal analysis:** Feature importance and model associations do not establish why a customer churns.
6. **Operational deployment:** Before deployment, the model would require monitoring, privacy and fairness review, and testing on more recent data.

### Overall conclusion

The analysis demonstrates a reproducible workflow for comparing two supervised classification methods on a commercially relevant problem. The preferred model should be selected from the observed test metrics in conjunction with the organisation's retention objective. Further temporal validation and a cost-benefit evaluation are required before the model could responsibly inform live retention decisions.

## Appendix A: Use of generative AI

This appendix records the use of generative AI in accordance with the assessment brief. Check and edit it so that it accurately describes the tools and prompts actually used.

### Tools used

- Gemini.
- ChatGPT.

### Examples of prompts

- "How should I structure a machine-learning notebook for predicting customer churn using Logistic Regression and a tuned Random Forest?"
- "How can I prevent data leakage by putting imputation, one-hot encoding and scaling inside a scikit-learn Pipeline and ColumnTransformer?"
- "How should I compare precision, recall, F1-score and ROC-AUC when the churn class is imbalanced?"
- "How can permutation importance be calculated for a fitted scikit-learn pipeline using the original input features?"
- "How should I explain false positives, false negatives and classification-threshold trade-offs in a customer-retention context?"

### Development, testing and integration

Generative AI was used for structural suggestions, explanations of machine-learning methods and assistance with code development. Suggestions were reviewed and adapted before inclusion. The notebook should be run from the first cell to the last in the intended JupyterHub environment. Outputs, metric values, plots, file paths and package compatibility should be checked by the student. Any code that was not tested should not be described as tested. The final interpretation and submission remain the student's responsibility.

*Before submission, retain only tools and prompts that genuinely reflect your use of AI, and follow any additional module or university guidance.*

## References

IBM (n.d.) *Telco Customer Churn*. Sample dataset distributed via Kaggle. Available at: https://www.kaggle.com/datasets/blastchar/telco-customer-churn (Accessed: 31 October 2026).

scikit-learn developers (n.d.) *scikit-learn: Machine Learning in Python — User Guide*. Available at: https://scikit-learn.org/stable/user_guide.html (Accessed: 31 October 2026).

scikit-learn developers (n.d.) *Pipeline and composite estimators*. Available at: https://scikit-learn.org/stable/modules/compose.html (Accessed: 31 October 2026).

scikit-learn developers (n.d.) *Permutation feature importance*. Available at: https://scikit-learn.org/stable/modules/permutation_importance.html (Accessed: 31 October 2026).